# MiMo-V2.5 Inference Optimization Toy Experiments

This notebook runs the companion Python file `mimo_optimization_experiments.py`.

It uses small PyTorch models and simulators to compare baseline vs optimized versions of several ideas from Xiaomi's MiMo-V2.5 inference blog: hybrid SWA KV cache, length bucketing, cache-affinity scheduling, encoder batching, and MTP-style speculative decode.

For Colab: upload this notebook and `mimo_optimization_experiments.py` into the same `/content` directory, then select a GPU runtime for the most interesting timing results.

In [ ]:
from pathlib import Path
import sys

script = Path('mimo_optimization_experiments.py')
if not script.exists():
    raise FileNotFoundError(
        'Upload mimo_optimization_experiments.py to the same directory as this notebook.'
    )

try:
    import torch
    print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
except ModuleNotFoundError:
    print('Installing PyTorch. Restart the runtime if Colab asks you to.')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])
    import torch
    print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())

## Quick Run

This should finish quickly and gives the main intuition. On CPU, some optimized kernels may not beat the baseline, but memory/work reductions are still visible. GPU timing usually shows the intended direction more clearly.

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable,
    'mimo_optimization_experiments.py',
    '--quick',
    '--device', 'auto',
    '--repeats', '5',
    '--warmup', '2',
    '--json', 'results_quick.json',
], check=True)

## Run Individual Experiments

Use this cell while studying one optimization at a time.

In [ ]:
# Change the name to one of:
# hybrid_swa, length_bucketing, scheduler, encoder_batching, mtp
experiment = 'hybrid_swa'
subprocess.run([
    sys.executable,
    'mimo_optimization_experiments.py',
    '--experiments', experiment,
    '--device', 'auto',
    '--quick',
    '--repeats', '7',
    '--warmup', '2',
], check=True)

## Less Tiny Run

This uses larger toy settings. It is better on a GPU runtime.

In [ ]:
subprocess.run([
    sys.executable,
    'mimo_optimization_experiments.py',
    '--device', 'auto',
    '--repeats', '5',
    '--warmup', '2',
    '--json', 'results_full.json',
], check=True)

## Inspect JSON Results

In [ ]:
import json
from pathlib import Path

path = Path('results_quick.json')
if path.exists():
    rows = json.loads(path.read_text())
    for row in rows:
        print(f"{row['experiment']:18s} {row['case']:14s} {row['metric']:24s} speedup={row['speedup']:.2f}x")